In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from dateutil import parser as dtparser
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [18]:
df=pd.read_csv('cleaned.csv')

In [19]:
df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'])

df = df.sort_values(by=['user_id', 'transaction_timestamp'])

df['time_diff'] = df.groupby('user_id')['transaction_timestamp'] \
    .diff().dt.total_seconds().fillna(0)

FEATURE 1 - Transaction Amount Deviation

In [20]:
df['user_avg_amount'] = df.groupby('user_id')['transaction_amount'].transform('mean')

df['amount_deviation'] = abs(df['transaction_amount'] - df['user_avg_amount'])

FEATURE 2 - Transaction Frequency

In [21]:
df['txn_count'] = df.groupby('user_id')['transaction_id'].transform('count')

FEATURE 3 - Rapid transaction


In [22]:
df['rapid_txn'] = (df['time_diff'] < 60).astype(int)

FEATURE 4 - Location change

In [23]:
df['location_change'] = (
    df.groupby('user_id')['user_location'].shift() != df['user_location']
).astype(int)

FEATURE 5 - Device change

In [24]:
df['device_change'] = (
    df.groupby('user_id')['device_id'].shift() != df['device_id']
).astype(int)

FEATURE 6 - Odd hour

In [25]:
df['odd_hour'] = ((df['hour'] < 6) | (df['hour'] > 23)).astype(int)

FEATURE 7 - Dormant account

In [26]:
df['dormant'] = (df['time_diff'] > 86400).astype(int)

In [27]:
features = [
    'transaction_amount',
    'rapid_txn',
    'location_change',
    'device_change',
    'odd_hour',
    'dormant',
    'amount_deviation',
    'txn_count'
]

In [28]:
df.to_csv('featured.csv', index=False)

In [29]:
df.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,...,is_fraud,time_diff,user_avg_amount,amount_deviation,txn_count,rapid_txn,location_change,device_change,odd_hour,dormant
1109,TXN-BFAB1556A909,USR0001,473.581395,2024-04-01 03:03:31.000000,19,37,1,177,0,0,...,0,0.000000,374.963838,98.617557,25,1,1,1,1,0
1166,TXN-BC3806284B2F,USR0001,373.233492,2024-04-01 13:07:18.827444,60,37,5,177,0,0,...,0,36227.827444,374.963838,1.730347,25,0,1,0,0,0
431,TXN-1CFEBE1835BA,USR0001,358.079800,2024-04-02 04:53:00.000000,60,13,5,177,0,2,...,0,56741.172556,374.963838,16.884038,25,0,0,0,1,0
321,TXN-B2F3A5F053BC,USR0001,358.000000,2024-04-04 02:44:00.000000,60,29,7,177,0,0,...,0,165060.000000,374.963838,16.963838,25,0,0,0,1,1
1158,TXN-8E4F07913EFA,USR0001,644.650000,2024-04-04 06:15:03.000000,17,15,4,176,0,0,...,0,12663.000000,374.963838,269.686162,25,0,1,1,0,0


In [30]:
df.isnull().sum()/len(df)*100

,0
transaction_id,0.0
user_id,0.0
transaction_amount,0.0
transaction_timestamp,0.0
user_location,0.0
merchant_location,0.0
merchant_category,0.0
device_id,0.0
device_type,0.0
payment_method,0.0
